# Walking EDA
Exploratory data analysis of my Apple Watch walking workouts. All cleaning and feature creation is done in `clean.ipynb`; this notebook only explores and visualizes.

1. Setup
2. Load & overview
3. Distributions: what does a typical walk look like?
4. Habits: when do I walk?
5. Trends over time: am I walking more?
6. Relationships & linear regression
7. Correlations: what moves together?
8. Categorical comparisons: how do walk types differ?
9. Fitness deep dive: am I getting fitter?
10. Multiple linear regression
11. Key findings

### 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.figsize": (9, 5)})

# Consistent colours and orders, reused by every chart
SEASON_ORDER = ["Spring", "Summer", "Fall", "Winter"]
SEASON_PALETTE = {"Spring": "#1baf7a", "Summer": "#eb6834", "Fall": "#4a3aa7", "Winter": "#2a78d6"}

YEAR_ORDER = [2024, 2025, 2026]
YEAR_PALETTE = dict(zip(YEAR_ORDER, sns.color_palette("crest", n_colors=3)))

WEEKEND_PALETTE = {"Weekday": "#a0a7b0", "Weekend": "#2a78d6"}
WEEKEND_LABELS = {False: "Weekday", True: "Weekend"}  # for hue=df["is_weekend"].map(WEEKEND_LABELS)

DAY_ORDER = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
TIME_ORDER = ["Morning", "Midday", "Afternoon", "Evening"]
BAND_ORDER = ["Short <2km", "Medium 2-4km", "Long 4km+"]

In [ ]:
def regression_stats(x, y, order=1):
    """Fit a polynomial with least squares and return its coefficients and R squared."""
    coefs = np.polyfit(x, y, order)
    predicted = np.polyval(coefs, x)
    r2 = 1 - np.sum((y - predicted) ** 2) / np.sum((y - np.mean(y)) ** 2)
    return coefs, r2


def add_regression_stats(ax, x, y, order=1, loc="upper left"):
    """Write the fitted equation and R squared in the corner of a chart."""
    coefs, r2 = regression_stats(x, y, order)
    if order == 1:
        label = f"y = {coefs[0]:.3g}x + {coefs[1]:.3g}\nR² = {r2:.2f}"
    else:
        label = f"order {order} fit\nR² = {r2:.2f}"
    x_pos, ha = (0.03, "left") if "left" in loc else (0.97, "right")
    y_pos, va = (0.95, "top") if "upper" in loc else (0.05, "bottom")
    ax.text(x_pos, y_pos, label, transform=ax.transAxes, ha=ha, va=va,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="#cccccc"))
    return coefs, r2

### 2. Load & overview

In [ ]:
df = pd.read_csv("../data/processed/clean_walks.csv")

# Dates are stored with local UTC offsets, so parse through UTC back to local time
for col in ["startDate", "endDate"]:
    df[col] = pd.to_datetime(df[col], utc=True).dt.tz_convert("America/Halifax")

df.head(2)

In [ ]:
df.shape

In [ ]:
df.describe().T.round(2)

In [ ]:
# Headline numbers
print(f"Total walks:     {len(df)}")
print(f"Total distance:  {df['distance_km'].sum():.1f} km")
print(f"Total time:      {df['duration'].sum() / 60:.1f} hours")
print(f"Total calories:  {df['calories_total'].sum():,.0f} kcal")
print(f"Longest walk:    {df['distance_km'].max():.2f} km")
print(f"Fastest walk:    {df['speed'].max():.2f} km/h")
print(f"Average walk:    {df['distance_km'].mean():.2f} km in {df['duration'].mean():.0f} min")

### 3. Distributions: what does a typical walk look like?
The shape of each core metric, and how it differs by season and year.

In [ ]:
metrics = {
    "distance_km": "Distance (km)",
    "duration": "Duration (min)",
    "speed": "Speed (km/h)",
    "heart_rate_average": "Average heart rate (bpm)",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, label) in zip(axes.flat, metrics.items()):
    sns.histplot(data=df, x=col, kde=True, color="#2a78d6", ax=ax)
    ax.axvline(df[col].median(), color="#333333", linestyle="--", linewidth=1, label="Median")
    ax.set_xlabel(label)
    ax.set_ylabel("Number of walks")
    ax.legend()
fig.suptitle("Distribution of Core Walk Metrics")
plt.tight_layout()

**Insight:** Distance and duration are strongly right skewed. The typical walk is about 2.1 km and 31 minutes, with a handful of long 7 to 12 km walks stretching the tail. Speed (median 4.3 km/h) and average heart rate (median 120 bpm) are much closer to symmetric, so the mean is a fair summary for those but the median is better for distance and duration.

In [ ]:
ax = sns.histplot(data=df, x="speed", hue="season", hue_order=SEASON_ORDER, palette=SEASON_PALETTE,
                  multiple="stack", binwidth=0.25)
ax.set_title("Speed Distribution by Season")
ax.set_xlabel("Speed (km/h)")
ax.set_ylabel("Number of walks")
plt.tight_layout()

In [ ]:
# Median speed per season, to back up the chart
df.groupby("season")["speed"].agg(["count", "median"]).loc[SEASON_ORDER].round(2)

**Insight:** Speed barely changes by season: medians range from 4.1 km/h (Summer) to 4.5 km/h (Winter). Summer has the most slow walks below 3 km/h, which fits heat and leisurely summer strolls. Winter and Fall have only 13 walks each, so treat their numbers with caution.

In [ ]:
ax = sns.ecdfplot(data=df, x="distance_km", hue="year", hue_order=YEAR_ORDER, palette=YEAR_PALETTE, linewidth=2)
ax.set_title("Cumulative Distribution of Walk Distance by Year")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Proportion of walks")
plt.tight_layout()

In [ ]:
df.groupby("year")["distance_km"].describe().round(2)

**Insight:** Walks have been getting longer every year. The median walk grew from 1.4 km (2024) to 2.1 km (2025) to 2.4 km (2026), and in 2026 about 27% of walks are over 4 km, compared with 10% in 2025 and 4% in 2024. The 2026 curve sits clearly to the right of the others.

In [ ]:
ax = sns.violinplot(data=df, x="season", y="heart_rate_average", order=SEASON_ORDER,
                    hue="season", palette=SEASON_PALETTE, inner="quart", cut=0, legend=False)
for violin in ax.collections:
    violin.set_alpha(0.35)
sns.stripplot(data=df, x="season", y="heart_rate_average", order=SEASON_ORDER,
              hue="season", palette=SEASON_PALETTE, jitter=0.15, size=6, edgecolor="white", linewidth=0.8,
              legend=False, ax=ax)
ax.set_title("Average Heart Rate by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Average heart rate (bpm)")
plt.tight_layout()

**Insight:** Summer (median 128 bpm) and Winter (128 bpm) walks run about 10 bpm higher than Spring (116 bpm) and Fall (118 bpm). The summer rise is likely heat; winter may be cold weather, or simply faster walking, since winter has the highest median speed.

### 4. Habits: when do I walk?
Which months, days and hours I walk most.

In [ ]:
ax = sns.countplot(data=df, x="month", hue="year", hue_order=YEAR_ORDER, palette=YEAR_PALETTE)
ax.set_title("Number of Walks by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Number of walks")
plt.tight_layout()

**Insight:** 2026 is by far the busiest year (62 of 106 walks), with a peak of 15 walks in May 2026. Earlier years had occasional bursts, like August 2024 with 10 walks. November through January are consistently quiet.

In [ ]:
ax = sns.countplot(data=df, x="day_name", order=DAY_ORDER, hue=df["is_weekend"].map(WEEKEND_LABELS),
                   hue_order=["Weekday", "Weekend"], palette=WEEKEND_PALETTE)
ax.set_title("Number of Walks by Day of Week")
ax.set_xlabel("Day of week")
ax.set_ylabel("Number of walks")
ax.get_legend().set_title("")
plt.tight_layout()

**Insight:** Saturday is the most popular day with 25 walks, almost 1 in 4. Monday is the least popular (10). Weekends make up 38% of walks despite being only 2 of 7 days.

In [ ]:
ax = sns.histplot(data=df, x="hour", hue=df["is_weekend"].map(WEEKEND_LABELS),
                  hue_order=["Weekday", "Weekend"], palette=WEEKEND_PALETTE,
                  binwidth=1, discrete=True, multiple="dodge", shrink=0.85)
ax.set_title("Walk Start Time: Weekday vs Weekend")
ax.set_xlabel("Hour of day (start)")
ax.set_ylabel("Number of walks")
ax.get_legend().set_title("")
plt.tight_layout()

**Insight:** Most walks start between 10:00 and 16:00. Weekday walks cluster around 10:00 and 14:00 to 15:00, like a morning break and an afternoon walk, while weekend walks peak at midday (12:00). Very early (before 8:00) and late (after 18:00) walks are rare.

In [ ]:
day_hour = (
    df.pivot_table(index="day_name", columns="hour", values="distance_km", aggfunc="size", fill_value=0)
    .reindex(DAY_ORDER)
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(day_hour, annot=True, fmt="d", cmap="Blues", linewidths=1, linecolor="white",
            cbar_kws={"label": "Number of walks"}, ax=ax)
ax.set_title("Walking Calendar: Day of Week vs Start Hour")
ax.set_xlabel("Hour of day (start)")
ax.set_ylabel("")
plt.tight_layout()

**Insight:** The two hot spots are Saturday at noon (7 walks) and Wednesday at 10:00 (6 walks). These look like established routines rather than random timing.

In [ ]:
year_month = (
    df.pivot_table(index="year", columns="month", values="distance_km", aggfunc="sum")
    .reindex(columns=range(1, 13))
)

fig, ax = plt.subplots(figsize=(12, 3.5))
sns.heatmap(year_month, annot=True, fmt=".0f", cmap="Blues", linewidths=1, linecolor="white",
            cbar_kws={"label": "Distance (km)"}, ax=ax)
ax.set_title("Total Distance Walked by Month and Year (km)")
ax.set_xticks(np.arange(12) + 0.5, ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()

**Insight:** February 2026 (41 km) and May 2026 (37 km) are the biggest months on record. Every month of 2026 so far beats the same month in 2024 and 2025, except August 2024.

### 5. Trends over time: am I walking more?
Distance and speed across the full history.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=df, x="startDate", y="cumulative_km", color="#a0a7b0", linewidth=2, ax=ax)
sns.scatterplot(data=df, x="startDate", y="cumulative_km", hue="season", hue_order=SEASON_ORDER,
                palette=SEASON_PALETTE, s=50, edgecolor="white", ax=ax)
ax.set_title("Cumulative Distance Walked")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative distance (km)")
plt.tight_layout()

**Insight:** Cumulative distance grew slowly through 2024 and 2025 (about 95 km in two years), then the curve steepened sharply in 2026, adding almost 200 km in 9 months. This is the headline story: 2026 has roughly doubled my walking.

In [ ]:
df["speed_rolling"] = df["speed"].rolling(5, min_periods=1).mean()  # plotting helper only

fig, ax = plt.subplots(figsize=(12, 5))
sns.scatterplot(data=df, x="startDate", y="speed", hue="distance_km", size="distance_km",
                sizes=(30, 250), palette="flare", edgecolor="white", ax=ax)
sns.lineplot(data=df, x="startDate", y="speed_rolling", color="#333333", linewidth=2, label="5 walk rolling average", ax=ax)
ax.set_title("Walking Speed Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Speed (km/h)")
ax.legend(title="Distance (km)", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

**Insight:** Speed is noisy walk to walk, with no strong long term trend. Median speed was 4.2 km/h in 2024, 3.8 km/h in 2025 and 4.5 km/h in 2026. Early 2026 has a run of fast, longer walks (large dark points), while the most recent walks have been slower.

In [ ]:
ax = sns.barplot(data=df, x="month", y="distance_km", hue="year", hue_order=YEAR_ORDER, palette=YEAR_PALETTE,
                 estimator="sum", errorbar=None)
ax.set_title("Total Distance by Month and Year")
ax.set_xlabel("Month")
ax.set_ylabel("Total distance (km)")
plt.tight_layout()

**Insight:** Looking at totals instead of counts makes 2026's jump even clearer: 195 km in 2026 so far, compared with 44 km in 2024 and 51 km in 2025. February 2026 stands out because it had several long walks, not just more walks.

### 6. Relationships & linear regression
Fitting straight lines (least squares) to the main relationships. Each chart shows the fitted equation and R², the share of variation explained by the line (0 to 1). The shaded band is the 95% confidence interval for the line.

In [ ]:
ax = sns.regplot(data=df, x="speed", y="heart_rate_average", color="#2a78d6",
                 scatter_kws={"s": 45, "edgecolor": "white"}, seed=42)
add_regression_stats(ax, df["speed"], df["heart_rate_average"])
ax.set_title("Average Heart Rate vs Speed")
ax.set_xlabel("Speed (km/h)")
ax.set_ylabel("Average heart rate (bpm)")
plt.tight_layout()

**Insight:** Heart rate rises by about 4.5 bpm for every extra 1 km/h of speed, as expected. But R² is only 0.11: speed explains just 11% of the variation in heart rate. Weather, terrain, walk length and day to day condition clearly matter more than speed alone.

In [ ]:
g = sns.lmplot(data=df, x="speed", y="heart_rate_average", hue="season", hue_order=SEASON_ORDER,
               palette=SEASON_PALETTE, ci=None, height=5, aspect=1.5, scatter_kws={"s": 45, "edgecolor": "white"})
g.ax.set_title("Heart Rate vs Speed by Season")
g.set_axis_labels("Speed (km/h)", "Average heart rate (bpm)")
g.legend.set_title("Season")

In [ ]:
# Slope and R² per season
pd.DataFrame(
    {season: dict(zip(["slope", "intercept"], regression_stats(g["speed"], g["heart_rate_average"])[0]),
                  r2=regression_stats(g["speed"], g["heart_rate_average"])[1], walks=len(g))
     for season, g in df.groupby("season")}
).T.loc[SEASON_ORDER].round(2)

**Insight:** All seasons show heart rate rising with speed. Summer's line sits highest, about 5 to 10 bpm above the others at the same speed, which supports the heat explanation. The steep Winter line (17 bpm per km/h, R² 0.64) is based on only 13 walks, so it may not hold up.

In [ ]:
g = sns.lmplot(data=df, x="speed", y="heart_rate_average", col="year", hue="year", palette=YEAR_PALETTE,
               height=4, aspect=1, scatter_kws={"s": 40, "edgecolor": "white"}, seed=42)
for ax, year in zip(g.axes.flat, g.col_names):
    subset = df[df["year"] == year]
    add_regression_stats(ax, subset["speed"], subset["heart_rate_average"])
g.set_axis_labels("Speed (km/h)", "Average heart rate (bpm)")
g.set_titles("{col_name}")
g.figure.suptitle("Heart Rate vs Speed by Year", y=1.04)

**Insight:** The slope is nearly identical each year (about 4.5 to 4.7 bpm per km/h), while the intercept drops from 104 (2024) to 101 (2026). In other words, the same speed now costs about 3 bpm less. This is a small shift, and with R² near 0.1 it is a hint of improved fitness rather than proof.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

sns.residplot(data=df, x="speed", y="heart_rate_average", color="#2a78d6",
              scatter_kws={"s": 40, "edgecolor": "white"}, ax=axes[0])
axes[0].set_title("Residuals of the Linear Fit")
axes[0].set_xlabel("Speed (km/h)")
axes[0].set_ylabel("Residual (bpm)")

for ax, order, color in [(axes[1], 1, "#2a78d6"), (axes[2], 2, "#eb6834")]:
    sns.regplot(data=df, x="speed", y="heart_rate_average", order=order, color=color,
                scatter_kws={"s": 40, "edgecolor": "white"}, ax=ax, seed=42)
    add_regression_stats(ax, df["speed"], df["heart_rate_average"], order=order)
    ax.set_title("Linear fit" if order == 1 else "Quadratic fit")
    ax.set_xlabel("Speed (km/h)")
    ax.set_ylabel("Average heart rate (bpm)")
plt.tight_layout()

**Insight:** The residuals are scattered evenly around zero with no clear curve, so a straight line is a reasonable model. The quadratic fit only raises R² from 0.11 to 0.13, which is not worth the extra complexity.

In [ ]:
g = sns.jointplot(data=df, x="distance_km", y="calories_total", kind="reg", color="#1baf7a",
                  height=6, joint_kws={"scatter_kws": {"s": 40, "edgecolor": "white"}, "seed": 42})
add_regression_stats(g.ax_joint, df["distance_km"], df["calories_total"])
g.set_axis_labels("Distance (km)", "Total calories (kcal)")
g.figure.suptitle("Calories Burned vs Distance", y=1.02)

**Insight:** This is the strongest relationship in the data (R² 0.92). Each km burns about 100 kcal, a clean rule of thumb and a good dashboard KPI. The points above the line at 7 km are longer, slower walks where basal calories add up over time.

In [ ]:
ax = sns.regplot(data=df, x="distance_km", y="duration", color="#4a3aa7",
                 scatter_kws={"s": 45, "edgecolor": "white"}, seed=42)
add_regression_stats(ax, df["distance_km"], df["duration"])
ax.set_title("Duration vs Distance")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Duration (min)")
plt.tight_layout()

**Insight:** Each km takes about 13.9 minutes (R² 0.83), which is a pace of roughly 4.3 km/h. Points above the line are slower, more leisurely walks. The long walks spread out more, since pace varies more over longer distances.

In [ ]:
ax = sns.scatterplot(data=df, x="speed", y="calories_per_km", hue="heart_rate_average", palette="coolwarm",
                     s=60, edgecolor="white")
add_regression_stats(ax, df["speed"], df["calories_per_km"], loc="upper right")
ax.set_title("Calories per km vs Speed (coloured by heart rate)")
ax.set_xlabel("Speed (km/h)")
ax.set_ylabel("Calories per km (kcal)")
ax.legend(title="Avg HR (bpm)", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

**Insight:** Faster walks burn fewer calories per km: about 12 kcal less per km for each extra 1 km/h (R² 0.29). That seems counterintuitive, but slower walks take longer per km, so more basal (resting) calories accumulate for the same distance. The biggest outlier (216 kcal/km) is a very short, slow walk.

### 7. Correlations: what moves together?
Pearson correlation between every pair of numeric walk metrics.

In [ ]:
corr_cols = ["distance_km", "duration", "speed", "calories_total", "calories_per_km",
             "heart_rate_average", "heart_rate_max", "heart_rate_range", "metres_per_beat", "walk_number"]
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
            linewidths=1, linecolor="white", cbar_kws={"label": "Correlation", "shrink": 0.8}, ax=ax)
ax.set_title("Correlation Between Walk Metrics")
plt.tight_layout()

**Insight:** Distance, duration and total calories move almost together (0.91 to 0.96). Speed is nearly independent of distance (0.08), so longer walks are not slower. Metres per beat is driven mostly by speed (0.88), while heart rate range grows with walk length (0.58), since longer walks have more chance of hills and variation.

In [ ]:
g = sns.pairplot(df, vars=["distance_km", "duration", "speed", "heart_rate_average", "calories_total"],
                 hue="season", hue_order=SEASON_ORDER, palette=SEASON_PALETTE, corner=True,
                 plot_kws={"s": 30, "edgecolor": "white"}, diag_kind="kde")
g.figure.suptitle("Pairwise Relationships by Season", y=1.02)

**Insight:** The season colours mostly overlap in every panel, so season is not a strong separator for most metrics. The exception is the heart rate diagonal, where the Summer curve is shifted to the right.

### 8. Categorical comparisons: how do walk types differ?
Comparing walks by length, time of day and season.

In [ ]:
ax = sns.boxplot(data=df, x="distance_band", y="speed", order=BAND_ORDER, color="#eeeeee",
                 showfliers=False, width=0.5)
sns.swarmplot(data=df, x="distance_band", y="speed", order=BAND_ORDER, hue="season",
              hue_order=SEASON_ORDER, palette=SEASON_PALETTE, size=6, ax=ax)
ax.set_title("Speed by Walk Length")
ax.set_xlabel("Walk length")
ax.set_ylabel("Speed (km/h)")
ax.legend(title="Season", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

**Insight:** Medium walks (2 to 4 km) are actually the slowest, with a median of 4.0 km/h. Long walks are the fastest (4.6 km/h), so going longer does not mean slowing down. The long walks are likely purposeful fitness walks, while medium walks are more casual.

In [ ]:
ax = sns.pointplot(data=df, x="time_of_day", y="heart_rate_average", order=TIME_ORDER,
                   hue=df["is_weekend"].map(WEEKEND_LABELS), hue_order=["Weekday", "Weekend"], palette=WEEKEND_PALETTE, dodge=0.3, capsize=0.1, markers=["o", "s"], linestyles=["-", "--"])
ax.set_title("Average Heart Rate by Time of Day (mean with 95% CI)")
ax.set_xlabel("Time of day")
ax.set_ylabel("Average heart rate (bpm)")
ax.get_legend().set_title("")
plt.tight_layout()

In [ ]:
pd.crosstab(df["time_of_day"], df["is_weekend"]).reindex(TIME_ORDER).rename(columns={False: "Weekday", True: "Weekend"})

**Insight:** Weekday heart rate is steady across the day (117 to 125 bpm). Weekend mornings show the highest average (133 bpm), but only 5 walks and a very wide confidence interval make it unreliable. Evening walks have the lowest heart rate, which fits relaxed evening strolls.

In [ ]:
g = sns.catplot(data=df, kind="bar", x="season", y="calories_per_km", order=SEASON_ORDER,
                col="distance_band", col_order=BAND_ORDER, hue="season", palette=SEASON_PALETTE,
                errorbar=("ci", 95), height=4, aspect=0.9, legend=False)
g.set_axis_labels("", "Calories per km (kcal)")
g.set_titles("{col_name}")
g.figure.suptitle("Calories per km by Season and Walk Length", y=1.04)

**Insight:** Summer has the highest calories per km in every length group, matching its higher heart rate. Most other bars are within about 10 kcal of each other, and several groups (such as Fall long walks, 1 walk) are too small for their error bars to mean much.

### 9. Fitness deep dive: am I getting fitter?
`metres_per_beat` is the distance covered per heartbeat. If fitness improves, I should cover more ground for each beat, or walk at the same speed with a lower heart rate.

In [ ]:
ax = sns.regplot(data=df, x="walk_number", y="metres_per_beat", color="#1baf7a",
                 scatter_kws={"s": 45, "edgecolor": "white"}, seed=42)
add_regression_stats(ax, df["walk_number"], df["metres_per_beat"])
ax.set_title("Cardio Efficiency Over Time")
ax.set_xlabel("Walk number")
ax.set_ylabel("Metres per heartbeat")
plt.tight_layout()

**Insight:** Metres per heartbeat has a slight upward slope (about +0.05 m per beat across all 106 walks), but R² is only 0.01, so there is no meaningful trend yet. Walk to walk variation is much larger than any long term fitness change.

In [ ]:
g = sns.lmplot(data=df, x="walk_number", y="heart_rate_average", hue="distance_band", hue_order=BAND_ORDER,
               palette=["#1baf7a", "#eb6834", "#4a3aa7"], ci=None, height=5, aspect=1.5,
               scatter_kws={"s": 40, "edgecolor": "white"})
g.ax.set_title("Heart Rate Over Time by Walk Length")
g.set_axis_labels("Walk number", "Average heart rate (bpm)")
g.legend.set_title("Walk length")

**Insight:** For medium and long walks, average heart rate trends downward over time (about 15 bpm lower across the dataset), while short walks stay flat. So there is a clearer fitness signal on longer walks, where effort is more sustained.

In [ ]:
# Only compare walks at a similar effort (4 to 5 km/h)
similar_speed = df[df["speed"].between(4, 5)]

ax = sns.boxplot(data=similar_speed, x="year", y="heart_rate_average", hue="year", palette=YEAR_PALETTE,
                 width=0.5, showfliers=False, legend=False)
sns.stripplot(data=similar_speed, x="year", y="heart_rate_average", color="#333333", size=5, jitter=0.12, ax=ax)
ax.set_title("Heart Rate at 4-5 km/h by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Average heart rate (bpm)")
plt.tight_layout()

similar_speed.groupby("year")["heart_rate_average"].agg(["count", "median"]).round(1)

**Insight:** Comparing only walks at a similar speed (4 to 5 km/h), 2026's median heart rate (117 bpm) is lower than 2025's (124 bpm) but higher than 2024's (113 bpm). With only 6 to 27 walks per year there is no clear improvement, so this does not confirm the fitness hint from the regression by year.

### 10. Multiple linear regression
Can heart rate be predicted better using several variables at once? Model: heart rate ~ speed + distance + season, fitted with least squares (`np.linalg.lstsq`). Season is one-hot encoded with Spring as the baseline.

In [ ]:
features = df[["speed", "distance_km", "season"]].astype({"season": pd.CategoricalDtype(SEASON_ORDER)})
X = pd.get_dummies(features, columns=["season"], drop_first=True, dtype=float)
X.insert(0, "intercept", 1.0)
y = df["heart_rate_average"]

coefs, *_ = np.linalg.lstsq(X.values, y.values, rcond=None)
df["hr_predicted"] = X.values @ coefs  # model output, used for the chart below

r2_multi = 1 - np.sum((y - df["hr_predicted"]) ** 2) / np.sum((y - y.mean()) ** 2)
r2_single = regression_stats(df["speed"], y)[1]

print(pd.Series(coefs, index=X.columns).round(2).to_string())
print(f"\nR² (speed only):             {r2_single:.3f}")
print(f"R² (speed + distance + season): {r2_multi:.3f}")

In [ ]:
ax = sns.scatterplot(data=df, x="hr_predicted", y="heart_rate_average", hue="season", hue_order=SEASON_ORDER,
                     palette=SEASON_PALETTE, s=55, edgecolor="white")
limits = [df[["hr_predicted", "heart_rate_average"]].min().min() - 2, df[["hr_predicted", "heart_rate_average"]].max().max() + 2]
ax.plot(limits, limits, color="#333333", linestyle="--", linewidth=1, label="Perfect prediction")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_title("Predicted vs Actual Heart Rate (multiple regression)")
ax.set_xlabel("Predicted average heart rate (bpm)")
ax.set_ylabel("Actual average heart rate (bpm)")
ax.legend()
plt.tight_layout()

**Insight:** Adding distance and season to speed doubles R² from 0.11 to 0.22, but the model still misses most of the variation (the points are loosely scattered around the dashed line). Summer adds about 9 bpm on top of Spring at the same speed and distance, which confirms the summer effect. Heart rate depends on things this dataset does not capture, such as temperature, hills and how rested I am.

### 11. Key findings
- **Walking has roughly doubled in 2026**: 62 walks and 195 km so far, compared with 44 km in 2024 and 51 km in 2025.
- **Walks are getting longer**: median distance grew from 1.4 km (2024) to 2.4 km (2026).
- **Saturday at noon is my walking routine**: Saturday is the top day (25 walks), and most walks start between 10:00 and 16:00.
- **About 100 kcal per km**: calories scale almost perfectly with distance (R² 0.92).
- **About 14 minutes per km**: duration scales tightly with distance (R² 0.83).
- **Speed is a weak predictor of heart rate**: +4.5 bpm per km/h, but R² is only 0.11. Adding distance and season only raises it to 0.22.
- **Summer walks are harder on the heart**: about 10 bpm higher than Spring at the same speed.
- **Fitness signal is weak**: HR on medium and long walks is trending down, and the HR vs speed line dropped about 3 bpm from 2024 to 2026, but efficiency (metres per beat) has no meaningful trend yet.

### Dashboard candidates
- KPI tiles: total walks, total km, total hours, total kcal, average kcal per km
- Cumulative distance over time (the growth story)
- Year × month distance heatmap
- Day of week × hour heatmap (walking routine)
- Heart rate vs speed regression, filterable by season and year
- Distance vs calories regression (kcal per km)
- Heart rate trend over time by walk length (fitness tracking)